# 06 — Estudo de Horizonte: degradação do desempenho (t+1 → t+12)

Justifica **empiricamente** a escolha do horizonte de previsão de 6 meses, medindo como o
desempenho dos modelos degrada conforme o horizonte cresce — e, principalmente, como evolui a
**vantagem do ML sobre a persistência ingênua** (`ŷ(t+h) = taxa(t)`).

**Hipótese**: em horizontes curtos a autocorrelação domina e a persistência empata com o ML
(problema trivial); em horizontes longos o sinal se dissipa para todos. O horizonte útil é o
intervalo em que o ML abre vantagem relevante *antes* da degradação geral.

## Protocolo (comparabilidade entre horizontes)
- **Janela de teste comum**: origens de 2022-01 a 2023-06 para *todos* os horizontes — assim
  até t+12 tem alvo observável (≤ 2024-06, limite da base) e todos os h são avaliados nos
  **mesmos meses de origem**.
- **Embargo variável**: o treino de cada horizonte termina h meses antes do teste (mesma lógica
  anti-vazamento do notebook 04).
- **Mesmas 28 variáveis** do pipeline principal: todas são retrospectivas (calculadas até t),
  logo independem do horizonte — apenas o alvo é reconstruído por segmento via `shift(-h)`.
- **Modelos**: Random Forest (campeão), XGBoost, Regressão Linear e persistência. Métrica: RMSE.
- **Verificação interna**: o alvo reconstruído para h=6 deve coincidir com o `target` original
  do dataset (mesma convenção do notebook 03).

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

BASE = Path().resolve().parent
PROC = BASE / 'data' / 'processed'
SEG_KEYS = ['cnae_secao', 'porte', 'modalidade']

with open(PROC / 'ml_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)
FEATURE_COLS = meta['feature_cols']

# --- Defasagem de publicacao (revisao A1) ---
# `lag_scr` e gravado pelo notebook 03. Quando > 0, as variaveis do SCR ja vem deslocadas e o
# "nivel corrente conhecido" na origem e `nivel_disp` = taxa(t - lag). O embargo de cada
# horizonte passa a ser h + lag: uma linha de treino em d so pode entrar se o seu alvo (d+h)
# ja estava PUBLICADO na origem, isto e, d + h + lag <= origem.
LAG_SCR = int(meta.get('lag_scr', 0))
NIVEL   = 'nivel_disp' if LAG_SCR else 'taxa_inadimplencia'

df = (
    pd.read_parquet(PROC / 'ml_dataset.parquet')
    .sort_values(SEG_KEYS + ['data_base'])
    .reset_index(drop=True)
)

# Reconstrucao do alvo por horizonte (shift(-h) POR SEGMENTO — mesma convencao do nb 03).
# O alvo e o valor REAL em t+h: a defasagem de publicacao afeta as variaveis, nao o alvo.
H_LIST = [1, 3, 6, 9, 12]
for h in H_LIST:
    df[f'target_h{h}'] = df.groupby(SEG_KEYS, observed=True)['taxa_inadimplencia'].shift(-h)

# Verificacao interna: h=6 reconstruido deve bater com o target original
chk = (df['target_h6'] - df['target']).abs()
n_cmp = chk.notna().sum()
print(f'Verificacao h=6 vs target original: {n_cmp:,} linhas comparaveis | '
      f'diff max = {chk.max():.2e} | iguais (<1e-12): {(chk < 1e-12).mean():.2%}')

TEST_START = pd.Timestamp('2022-01-01')
TEST_END   = pd.Timestamp('2023-06-30')   # origem max p/ h=12 ter alvo dentro da base (2024-06)
print(f'\nDefasagem de publicacao do SCR: {LAG_SCR} mes(es)  |  nivel conhecido na origem: {NIVEL}')
print(f'Janela de teste comum (origens): {TEST_START.date()} -> {TEST_END.date()} '
      f'({(TEST_END.year-TEST_START.year)*12 + TEST_END.month-TEST_START.month + 1} meses)')

## 1. Treino e avaliação por horizonte

Para cada h: descarta linhas sem alvo, aplica embargo de h meses, treina os três modelos de ML
e avalia todos (incl. persistência) nas mesmas origens de teste.

In [ ]:
def make_models():
    return {
        'RandomForest': Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('reg', RandomForestRegressor(n_estimators=200, max_depth=12,
                                          min_samples_leaf=10, n_jobs=-1, random_state=42)),
        ]),
        'XGBoost': Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('reg', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                 subsample=0.8, colsample_bytree=0.8,
                                 random_state=42, verbosity=0)),
        ]),
        'Reg. Linear': Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scl', StandardScaler()),
            ('reg', LinearRegression()),
        ]),
    }

rows = []
for h in H_LIST:
    tgt = f'target_h{h}'
    d  = df.dropna(subset=[tgt])
    # embargo = h + defasagem: o alvo da linha de treino precisa estar PUBLICADO na origem
    embargo_lim = TEST_START - pd.DateOffset(months=h + LAG_SCR)
    tr = d[d['data_base'] < embargo_lim]
    te = d[(d['data_base'] >= TEST_START) & (d['data_base'] <= TEST_END)]

    Xtr, ytr = tr[FEATURE_COLS], tr[tgt].to_numpy()
    Xte, yte = te[FEATURE_COLS], te[tgt].to_numpy()

    res = {'h': h, 'n_treino': len(tr), 'n_teste': len(te)}
    # persistencia = repetir o ultimo valor CONHECIDO na origem
    res['Persistencia'] = float(np.sqrt(mean_squared_error(yte, te[NIVEL].to_numpy())))
    for nome, pipe in make_models().items():
        pipe.fit(Xtr, ytr)
        res[nome] = float(np.sqrt(mean_squared_error(yte, pipe.predict(Xte))))
    res['Ganho RF (%)'] = (res['Persistencia'] - res['RandomForest']) / res['Persistencia'] * 100
    rows.append(res)
    print(f"h={h:2d}  treino={len(tr):>6,}  teste={len(te):>6,}  "
          f"RF={res['RandomForest']:.4f}  XGB={res['XGBoost']:.4f}  "
          f"LinReg={res['Reg. Linear']:.4f}  Persist={res['Persistencia']:.4f}  "
          f"ganho RF={res['Ganho RF (%)']:+.1f}%")

df_hor = pd.DataFrame(rows).set_index('h')
print('\n=== RMSE por horizonte (janela de teste comum 2022-01..2023-06) ===')
cols_show = ['RandomForest', 'XGBoost', 'Reg. Linear', 'Persistencia', 'Ganho RF (%)']
print(df_hor[cols_show].round(4).to_string())
df_hor.to_csv(PROC / 'resultados_horizonte.csv')
print('\nSalvo: resultados_horizonte.csv')

## 2. Curva de degradação

Painel esquerdo: RMSE de cada modelo em função do horizonte (menor = melhor).
Painel direito: vantagem percentual do Random Forest sobre a persistência — a leitura-chave
para a escolha do horizonte.

In [ ]:
# Paleta categorica validada (dataviz skill — ordem fixa dos slots, CVD dE 24.2)
COL = {
    'RandomForest': '#2a78d6',   # slot 1 azul
    'XGBoost':      '#1baf7a',   # slot 2 aqua
    'Reg. Linear':  '#eda100',   # slot 3 amarelo
    'Persistencia': '#008300',   # slot 4 verde (baseline — tracejado)
}
SURFACE, GRID, AXIS  = '#fcfcfb', '#e1e0d9', '#c3c2b7'
INK, INK2, MUTED     = '#0b0b0b', '#52514e', '#898781'

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), facecolor=SURFACE)

# --- Painel A: RMSE vs horizonte ---
ax = axes[0]
ax.set_facecolor(SURFACE)
for nome in ['RandomForest', 'XGBoost', 'Reg. Linear', 'Persistencia']:
    ls = '--' if nome == 'Persistencia' else '-'
    ax.plot(df_hor.index, df_hor[nome], ls, color=COL[nome], linewidth=2,
            marker='o', markersize=6, markerfacecolor=COL[nome],
            markeredgecolor=SURFACE, markeredgewidth=1.2, label=nome)
    # rotulo direto no fim da linha (texto em tinta, nao na cor da serie)
    ax.annotate(nome, (df_hor.index[-1], df_hor[nome].iloc[-1]),
                xytext=(7, 0), textcoords='offset points',
                fontsize=8.5, color=INK2, va='center')
ax.set_xticks(list(df_hor.index))
ax.set_xlabel('Horizonte de previsão (meses)', fontsize=10, color=INK2)
ax.set_ylabel('RMSE', fontsize=10, color=INK2)
ax.set_title('Degradação do RMSE por horizonte', fontsize=11.5, color=INK, pad=10)
ax.grid(True, axis='y', color=GRID, linewidth=0.7)
ax.legend(fontsize=8.5, frameon=False, loc='upper left', labelcolor=INK2)
ax.set_xlim(0.3, 15.6)
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
for s in ['left', 'bottom']:
    ax.spines[s].set_color(AXIS)
ax.tick_params(colors=MUTED, labelsize=9)

# --- Painel B: ganho do RF sobre a persistencia ---
ax = axes[1]
ax.set_facecolor(SURFACE)
g = df_hor['Ganho RF (%)']
ax.plot(df_hor.index, g, '-', color=COL['RandomForest'], linewidth=2,
        marker='o', markersize=7, markerfacecolor=COL['RandomForest'],
        markeredgecolor=SURFACE, markeredgewidth=1.2)
for h, v in g.items():
    ax.annotate(f'{v:+.1f}%', (h, v), xytext=(0, 9), textcoords='offset points',
                ha='center', fontsize=8.5, color=INK2)
ax.axhline(0, color=AXIS, linewidth=1)
ax.axvline(6, color=MUTED, linewidth=1, linestyle=':', alpha=0.8)
ax.annotate('horizonte\nadotado (t+6)', (6, ax.get_ylim()[0]), xytext=(10, 12),
            textcoords='offset points', fontsize=8.5, color=MUTED)
ax.set_xticks(list(df_hor.index))
ax.set_xlabel('Horizonte de previsão (meses)', fontsize=10, color=INK2)
ax.set_ylabel('Redução de RMSE vs persistência (%)', fontsize=10, color=INK2)
ax.set_title('Vantagem do Random Forest sobre a persistência', fontsize=11.5, color=INK, pad=10)
ax.grid(True, axis='y', color=GRID, linewidth=0.7)
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
for s in ['left', 'bottom']:
    ax.spines[s].set_color(AXIS)
ax.tick_params(colors=MUTED, labelsize=9)

plt.tight_layout()
plt.savefig(PROC / 'resultados_horizonte.png', dpi=150, bbox_inches='tight', facecolor=SURFACE)
plt.show()
print('Salvo: resultados_horizonte.png')